In [3]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# ============================================================
# 1. LOAD  DATASET
# ============================================================

url = "https://raw.githubusercontent.com/sridevi-tandley1/APA/main/data/loan_applications.csv"

df = pd.read_csv(url)

print("DATASET SHAPE:", df.shape)
print("\nAVAILABLE COLUMNS:")
print(df.columns.tolist())


# ============================================================
# 2. DEFINE FEATURES AND TARGET
# ============================================================

features = ["income", "credit_score", "pin_band"]
target = "approved"

X = df[features]
y = df[target]

# Split data for evaluation
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X,
    y,
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

train_data = df.loc[train_idx].copy()
test_data = df.loc[test_idx].copy()

# Reweight gender groups using training data
weights = (
    train_data["gender"]
    .map(1 / train_data["gender"].value_counts(normalize=True))
    .to_numpy()
)


# ============================================================
# 3. DEFINE MODELS
# ============================================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        max_depth=3,
        random_state=42
    )
}


# ============================================================
# 4. EVALUATION FUNCTION
# ============================================================

results = []

def evaluate_model(model, X_train, X_test, y_train, y_test,
                   test_data, model_name, mitigation, sample_weight=None):

    model.fit(X_train, y_train, sample_weight=sample_weight)

    predictions = model.predict(X_test)

    evaluation_data = test_data.copy()
    evaluation_data["pred"] = predictions

    group_rates = evaluation_data.groupby("gender")["pred"].mean()

    female_rate = group_rates.get("F", float("nan"))
    male_rate = group_rates.get("M", float("nan"))

    approval_gap = abs(female_rate - male_rate) * 100

    accuracy = accuracy_score(y_test, predictions) * 100

    results.append({
        "Model": model_name,
        "Mitigation": mitigation,
        "Approval Gap (pp)": approval_gap,
        "Accuracy (%)": accuracy,
        "Group Approval Rates": {
            "F": female_rate,
            "M": male_rate
        }
    })


# ============================================================
# 5. RUN MODELS WITH THREE CONDITIONS
# ============================================================

for name, base_model in models.items():

    # ORIGINAL MODEL
    evaluate_model(
        base_model.__class__(**base_model.get_params()),
        X_train,
        X_test,
        y_train,
        y_test,
        test_data,
        name,
        "Original"
    )

    # MITIGATION 1: REWEIGHTING
    evaluate_model(
        base_model.__class__(**base_model.get_params()),
        X_train,
        X_test,
        y_train,
        y_test,
        test_data,
        name,
        "Reweighting",
        sample_weight=weights
    )

    # MITIGATION 2: REWEIGHTING + REMOVE PIN BAND
    reduced_features = ["income", "credit_score"]

    evaluate_model(
        base_model.__class__(**base_model.get_params()),
        X_train[reduced_features],
        X_test[reduced_features],
        y_train,
        y_test,
        test_data,
        name,
        "Reweight + Remove PIN",
        sample_weight=weights
    )


# ============================================================
# 6. MODEL COMPARISON - COMPLETE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("MODEL COMPARISON - COMPLETE RESULTS")
print("=" * 75)

display(results_df)


# ============================================================
# 7. TABLE 1 - APPROVAL GAP
# ============================================================

gap_table = results_df.pivot(
    index="Model",
    columns="Mitigation",
    values="Approval Gap (pp)"
).round(2)

print("\nTABLE 1: APPROVAL GAP (PERCENTAGE POINTS)")
display(gap_table)


# ============================================================
# 8. TABLE 2 - ACCURACY
# ============================================================

accuracy_table = results_df.pivot(
    index="Model",
    columns="Mitigation",
    values="Accuracy (%)"
).round(2)

print("\nTABLE 2: MODEL ACCURACY (%)")
display(accuracy_table)


# ============================================================
# 9. TABLE 3 - GROUP-WISE APPROVAL RATES
# ============================================================

group_rows = []

for _, row in results_df.iterrows():

    rates = row["Group Approval Rates"]

    group_rows.append({
        "Model": row["Model"],
        "Mitigation": row["Mitigation"],
        "Female (%)": round(rates["F"] * 100, 2),
        "Male (%)": round(rates["M"] * 100, 2)
    })

group_table = pd.DataFrame(group_rows)

print("\nTABLE 3: GROUP-WISE APPROVAL RATES (%)")
display(group_table)


# ============================================================
# 10. TABLE 4 - HISTORICAL APPROVAL RATES
# ============================================================

historical_table = (
    df.groupby("gender")["approved"]
    .mean()
    .mul(100)
    .reset_index()
)

historical_table.columns = [
    "Gender",
    "Historical Approval Rate (%)"
]

historical_table["Historical Approval Rate (%)"] = (
    historical_table["Historical Approval Rate (%)"].round(2)
)

print("\nTABLE 4: HISTORICAL APPROVAL RATES")
display(historical_table)


# ============================================================
# 11. TABLE 5 - PROXY CORRELATIONS
# ============================================================

proxy_table = (
    df.corr(numeric_only=True)["is_female"]
    .drop("is_female")
    .sort_values()
    .reset_index()
)

proxy_table.columns = [
    "Feature",
    "Correlation with is_female"
]

proxy_table["Correlation with is_female"] = (
    proxy_table["Correlation with is_female"].round(4)
)

print("\nTABLE 5: PROXY CORRELATIONS")
display(proxy_table)


# ============================================================
# 12. EXPORT ALL TABLES TO EXCEL
# ============================================================

with pd.ExcelWriter("Loan_Model_Slide_Tables.xlsx") as writer:

    gap_table.to_excel(writer, sheet_name="Approval Gap")
    accuracy_table.to_excel(writer, sheet_name="Accuracy")

    group_table.to_excel(
        writer,
        sheet_name="Group Approval Rates",
        index=False
    )

    historical_table.to_excel(
        writer,
        sheet_name="Historical Rates",
        index=False
    )

    proxy_table.to_excel(
        writer,
        sheet_name="Proxy Correlations",
        index=False
    )

print("\nSUCCESS: All five tables exported to Loan_Model_Slide_Tables.xlsx")


DATASET SHAPE: (2000, 8)

AVAILABLE COLUMNS:
['applicant_id', 'gender', 'is_female', 'age', 'income', 'credit_score', 'pin_band', 'approved']

MODEL COMPARISON - COMPLETE RESULTS


,Model,Mitigation,Approval Gap (pp),Accuracy (%),Group Approval Rates
0,Logistic Regression,Original,7.340153,79.25,"{'F': 0.8352941176470589, 'M': 0.908695652173913}"
1,Logistic Regression,Reweighting,6.035806,79.00,"{'F': 0.8352941176470589, 'M': 0.8956521739130..."
2,Logistic Regression,Reweight + Remove PIN,9.641944,79.25,"{'F': 0.8470588235294118, 'M': 0.9434782608695..."
3,Decision Tree,Original,5.601023,77.25,"{'F': 0.8352941176470589, 'M': 0.8913043478260..."
4,Decision Tree,Reweighting,4.424552,75.75,"{'F': 0.8470588235294118, 'M': 0.8913043478260..."
5,Decision Tree,Reweight + Remove PIN,9.411765,77.50,"{'F': 0.8058823529411765, 'M': 0.9}"
6,Random Forest,Original,9.769821,79.50,"{'F': 0.8588235294117647, 'M': 0.9565217391304..."
7,Random Forest,Reweighting,9.335038,79.25,"{'F': 0.8588235294117647, 'M': 0.9521739130434..."
8,Random Forest,Reweight + Remove PIN,9.667519,80.00,"{'F': 0.8294117647058824, 'M': 0.9260869565217..."
9,Gradient Boosting,Original,9.079284,78.25,"{'F': 0.8352941176470589, 'M': 0.9260869565217..."



TABLE 1: APPROVAL GAP (PERCENTAGE POINTS)


Mitigation,Original,Reweight + Remove PIN,Reweighting
Model,,,
Decision Tree,5.60,9.41,4.42
Gradient Boosting,9.08,7.62,7.93
Logistic Regression,7.34,9.64,6.04
Random Forest,9.77,9.67,9.34



TABLE 2: MODEL ACCURACY (%)


Mitigation,Original,Reweight + Remove PIN,Reweighting
Model,,,
Decision Tree,77.25,77.50,75.75
Gradient Boosting,78.25,78.50,78.50
Logistic Regression,79.25,79.25,79.00
Random Forest,79.50,80.00,79.25


... (3 outputs left out)

,Gender,Historical Approval Rate (%)
0,F,71.91
1,M,80.91



TABLE 5: PROXY CORRELATIONS


,Feature,Correlation with is_female
0,pin_band,-0.1368
1,income,-0.1323
2,approved,-0.1047
3,credit_score,0.0030
4,applicant_id,0.0211
5,age,0.0380



SUCCESS: All five tables exported to Loan_Model_Slide_Tables.xlsx
